# Strategy Runner (LIVE)

This notebook is a thin wrapper around the production trading runner.

It uses:
1. The cached discovery output in `data/latest_discovery.json`.
2. The same trade gating and execution code used by GitHub Actions.
3. A safe wrapper that treats missing credentials as a clean skip instead of a notebook crash.

In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "run_trade_from_cache.py").exists() or (candidate / "run_discovery_cache.py").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")

repo_root = find_repo_root(Path.cwd())
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Repository root: {repo_root}")

In [ ]:
import os

os.environ['TRADING_MODE'] = 'LIVE'
os.environ['DISCOVERY_CACHE_PATH'] = 'data/latest_discovery.json'
os.environ['DISCOVERY_MAX_AGE_DAYS'] = '10'

module_name = 'run_trade_from_cache'
try:
    module = __import__(module_name, fromlist=["main"])
except ModuleNotFoundError as exc:
    print(f"Runner dependency missing: {exc}")
    module = None

if module is not None:
    try:
        module.main()
    except SystemExit as exc:
        if exc.code not in (0, None):
            raise
        print(f"Runner exited cleanly with code {exc.code}.")